# Full Pipeline — Deblur & Denoise on LBC Cervical Images

End-to-end walkthrough from degradation through all three restoration methods to comparison.

**Group R** — Francesco Castaldi, Paolo Fusco

**Methods:** TV (variational), UNet (end-to-end), DiffPIR (generative)

## 1. Setup

Load configuration and helper imports.

In [ ]:
import sys
sys.path.insert(0, "..")

import torch
import matplotlib.pyplot as plt
import numpy as np
from pathlib import Path

from src.data.dataset import load_config, LBCDataset
from src.degradation.degradation import degrade
from src.methods.tv.tv import tv_restore
from src.methods.unet.unet import UNet
from src.methods.diffpir.diffpir import run_diffpir
from src.eval.metrics import evaluate
from src.plots.visualize import show_comparison, plot_metrics

config = load_config()
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")
print(f"Config seed: {config['seed']}")
print(f"Image size: {config['dataset']['image_size']}")
print(f"Noise levels: {config['degradation']['noise_levels']}")
print(f"Blur: sigma={config['degradation']['blur_sigma']}, kernel={config['degradation']['kernel_size']}")

## 2. Dataset

Load the LBC cervical cancer dataset and inspect a sample image.

In [ ]:
test_dataset = LBCDataset("data/splits/test.txt", config["dataset"]["image_size"])
print(f"Test set size: {len(test_dataset)} images")

# Load a single test image
gt = test_dataset[0]  # shape: [3, 256, 256], range [-1, 1]
print(f"GT shape: {gt.shape}")
print(f"GT range: [{gt.min().item():.3f}, {gt.max().item():.3f}]")

In [ ]:
def to_display(tensor):
    """Convert [-1, 1] tensor to [0, 1] numpy for display."""
    return (tensor * 0.5 + 0.5).clamp(0, 1).permute(1, 2, 0).numpy()

plt.figure(figsize=(4, 4))
plt.imshow(to_display(gt))
plt.title("Ground Truth")
plt.axis("off")
plt.show()

## 3. Degradation Pipeline

Apply Gaussian blur (σ=2, kernel 9×9) + AWGN at 4 noise levels.

This is the **shared degradation** — every method receives the same degraded input.

In [ ]:
noise_levels = config["degradation"]["noise_levels"]
kernel_size = config["degradation"]["kernel_size"]
blur_sigma = config["degradation"]["blur_sigma"]

degraded_images = {}
fig, axes = plt.subplots(1, len(noise_levels) + 1, figsize=(16, 4))
axes[0].imshow(to_display(gt))
axes[0].set_title("Original (GT)")
axes[0].axis("off")

for i, sigma_n in enumerate(noise_levels):
    degraded = degrade(gt, kernel_size=kernel_size, sigma=blur_sigma, noise_level=sigma_n)
    degraded_images[sigma_n] = degraded
    psnr_deg = evaluate(degraded, gt)["psnr"]
    axes[i + 1].imshow(to_display(degraded))
    axes[i + 1].set_title(f"σₙ={sigma_n}\nPSNR={psnr_deg:.2f} dB")
    axes[i + 1].axis("off")

plt.suptitle("Degradation: Gaussian Blur (σ=2, k=9) + AWGN", fontsize=14)
plt.tight_layout()
plt.show()

## 4. Method 1 — Total Variation (Variational)

Minimizes: `||H*x - y||² + λ_reg * TV(x)`

| Parameter | Value |
|---|---|
| λ_reg | 0.005 |
| Iterations | 150 |
| Optimizer | Adam (lr=0.001) |

In [ ]:
lambda_reg = config["tv"]["lambda_reg"]
max_iter = config["tv"]["max_iter"]

tv_results = {}
print("Running TV restoration on all noise levels...")
for sigma_n in noise_levels:
    restored = tv_restore(
        degraded_images[sigma_n],
        kernel_size=kernel_size,
        sigma=blur_sigma,
        lambda_reg=lambda_reg,
        max_iter=max_iter,
    )
    m = evaluate(restored, gt)
    tv_results[sigma_n] = (restored, m)
    print(f"  σₙ={sigma_n}: PSNR={m['psnr']:.2f} dB, SSIM={m['ssim']:.4f}")

### TV Qualitative Results

In [ ]:
fig, axes = plt.subplots(1, len(noise_levels), figsize=(16, 4))
for i, sigma_n in enumerate(noise_levels):
    restored, m = tv_results[sigma_n]
    axes[i].imshow(to_display(restored))
    axes[i].set_title(f"σₙ={sigma_n}\nPSNR={m['psnr']:.2f}\nSSIM={m['ssim']:.4f}")
    axes[i].axis("off")
plt.suptitle("TV Restoration Results", fontsize=14)
plt.tight_layout()
plt.show()

## 5. Method 2 — UNet (End-to-End Deep Learning)

Optimized architecture with noise conditioning. Trained for 50 epochs on CPU.

| Component | Detail |
|---|---|
| Architecture | Encoder-decoder with skip connections |
| Params | ~1.9M |
| Input | 4 channels (RGB + noise map) |
| Loss | L1 |
| Training | Multi-noise augmentation (σ random per batch) |

In [ ]:
unet_weights = Path(config["eval"]["results_dir"]) / "unet" / "best_model.pth"
unet_available = unet_weights.exists()

if unet_available:
    from src.methods.unet.unet import UNet
    import torch.nn.functional as F
    
    model = UNet(in_channels=4, out_channels=3, features=(16, 32, 64, 128)).to(device)
    model.load_state_dict(torch.load(unet_weights, map_location=device, weights_only=True))
    model.eval()
    print(f"UNet model loaded from {unet_weights}")
    
    def make_noise_map(batch_size, image_size, sigma_n, device):
        return torch.full((batch_size, 1, image_size, image_size), sigma_n, device=device)
    
    unet_results = {}
    print("\nRunning UNet restoration...")
    for sigma_n in noise_levels:
        degraded = degraded_images[sigma_n].unsqueeze(0).to(device)
        noise_map = make_noise_map(1, config["dataset"]["image_size"], sigma_n, device)
        model_input = torch.cat([degraded, noise_map], dim=1)
        with torch.no_grad():
            restored = model(model_input).squeeze(0).cpu()
        m = evaluate(restored, gt)
        unet_results[sigma_n] = (restored, m)
        print(f"  σₙ={sigma_n}: PSNR={m['psnr']:.2f} dB, SSIM={m['ssim']:.4f}")
else:
    print("UNet weights not found. Train first with: python scripts/run_unet.py")
    unet_results = None

### UNet Qualitative Results

In [ ]:
if unet_results:
    fig, axes = plt.subplots(1, len(noise_levels), figsize=(16, 4))
    for i, sigma_n in enumerate(noise_levels):
        restored, m = unet_results[sigma_n]
        axes[i].imshow(to_display(restored))
        axes[i].set_title(f"σₙ={sigma_n}\nPSNR={m['psnr']:.2f}\nSSIM={m['ssim']:.4f}")
        axes[i].axis("off")
    plt.suptitle("UNet Restoration Results", fontsize=14)
    plt.tight_layout()
    plt.show()
else:
    plt.text(0.5, 0.5, "UNet not available — train the model first", ha='center', fontsize=14)
    plt.show()

## 6. Method 3 — DiffPIR (Generative)

Diffusion-based posterior sampling with FFT data-fidelity.

| Parameter | Value |
|---|---|
| t_start | 50 |
| num_steps | 15 |
| λ | 10.0 |
| ζ | 0.0 (DDIM, deterministic) |
| Denoiser | LightUNet (1.26M params) |

In [ ]:
weights_path = Path(config["diffpir"]["weights"])
diffpir_available = weights_path.exists()

if diffpir_available:
    diffpir_results = {}
    print("Running DiffPIR restoration...")
    for sigma_n in noise_levels:
        restored, elapsed = run_diffpir(
            degraded_images[sigma_n],
            num_steps=config["diffpir"]["num_steps"],
            noise_level=sigma_n,
            weights_path=weights_path,
            kernel_size=kernel_size,
            blur_sigma=blur_sigma,
            lambda_=config["diffpir"]["lambda"],
            zeta=config["diffpir"]["zeta"],
            t_start=config["diffpir"]["t_start"],
            return_timing=True,
        )
        m = evaluate(restored, gt)
        diffpir_results[sigma_n] = (restored, m, elapsed)
        print(f"  σₙ={sigma_n}: PSNR={m['psnr']:.2f} dB, SSIM={m['ssim']:.4f}, time={elapsed:.1f}s")
else:
    print("DiffPIR weights not found at", weights_path)
    diffpir_results = None

### DiffPIR Qualitative Results

In [ ]:
if diffpir_results:
    fig, axes = plt.subplots(1, len(noise_levels), figsize=(16, 4))
    for i, sigma_n in enumerate(noise_levels):
        restored, m, elapsed = diffpir_results[sigma_n]
        axes[i].imshow(to_display(restored))
        axes[i].set_title(f"σₙ={sigma_n}\nPSNR={m['psnr']:.2f}\nSSIM={m['ssim']:.4f}\n{elapsed:.1f}s")
        axes[i].axis("off")
    plt.suptitle("DiffPIR Restoration Results", fontsize=14)
    plt.tight_layout()
    plt.show()
else:
    plt.text(0.5, 0.5, "DiffPIR not available — train the DDPM first", ha='center', fontsize=14)
    plt.show()

## 7. Comparative Results

Side-by-side comparison of all three methods at each noise level.

In [ ]:
methods_available = {
    "TV": tv_results,
    "UNet": unet_results,
    "DiffPIR": diffpir_results,
}

for sigma_n in noise_levels:
    fig, axes = plt.subplots(1, 4, figsize=(16, 4))
    
    # Ground truth
    axes[0].imshow(to_display(gt))
    axes[0].set_title("Ground Truth")
    axes[0].axis("off")
    
    # Degraded
    axes[1].imshow(to_display(degraded_images[sigma_n]))
    deg_m = evaluate(degraded_images[sigma_n], gt)
    axes[1].set_title(f"Degraded\nPSNR={deg_m['psnr']:.1f}")
    axes[1].axis("off")
    
    # Methods
    for j, (name, results) in enumerate(methods_available.items()):
        if results and sigma_n in results:
            restored, m = results[sigma_n][:2]
            axes[j + 2].imshow(to_display(restored))
            axes[j + 2].set_title(f"{name}\nPSNR={m['psnr']:.1f}\nSSIM={m['ssim']:.3f}")
        axes[j + 2].axis("off")
    
    plt.suptitle(f"Comparison at σₙ={sigma_n}", fontsize=14)
    plt.tight_layout()
    plt.show()

### Quantitative Summary

Metrics table for all methods across all noise levels.

In [ ]:
import pandas as pd

rows = []
for sigma_n in noise_levels:
    deg_m = evaluate(degraded_images[sigma_n], gt)
    row = {"σₙ": sigma_n, "Metric": "Degraded", "PSNR (dB)": f"{deg_m['psnr']:.2f}", "SSIM": f"{deg_m['ssim']:.4f}"}
    rows.append(row)
    for name, results in methods_available.items():
        if results and sigma_n in results:
            m = results[sigma_n][1]
            row = {"σₙ": sigma_n, "Metric": name, "PSNR (dB)": f"{m['psnr']:.2f}", "SSIM": f"{m['ssim']:.4f}"}
            rows.append(row)

df = pd.DataFrame(rows)
print(df.to_string(index=False))

## 8. Key Observations

### TV (Variational)
- **Excels at low noise**: PSNR > 32 dB for σₙ ≤ 0.01
- **Degrades at high noise**: staircasing artifacts, fixed λ not optimal
- No training required, interpretable

### UNet (End-to-End)
- **Robust across all noise levels**: only ~1 dB drop from σₙ=0.005 to 0.1
- **Fastest inference**: ~0.035 s/image
- **Best at high noise**: 28.93 dB vs TV 26.54 dB at σₙ=0.1

### DiffPIR (Generative)
- **Improves with more noise**: PSNR increases from 16.67 dB (σₙ=0.005) to 24.68 dB (σₙ=0.1)
- **Hallucinates at low noise**: adds artificial textures when signal is clean
- **Slowest**: ~2 s/image

### Complementary Strengths
- No single method dominates all regimes — the choice depends on the noise level and latency requirements